![Banner](../Image/04_DeepCuaslaML.png)

# 4. Time Series Deep Causal Machine Learning

Most of the earlier chapters treat observations as **i.i.d.** — each row is an independent unit, and causal questions are about *treatments applied to units*. Time series break both assumptions. Observations are ordered and autocorrelated, a "unit" is often a single evolving system, and the causal questions change shape: instead of "what is the effect of treatment on this population?", we ask

- **Which series drives which, and at what lag?** (temporal causal discovery)
- **What would this trajectory have looked like under a different intervention?** (counterfactual forecasting)
- **What is the effect of an intervention that unfolds over time?** (dynamic treatment effects)

This chapter surveys the deep-learning model families that answer these questions, and grounds each one in the corresponding estimator in **PyDeepCausalML** (`pydeepcausalml.timeseries`). The theory below is the *why*; the package is the *how*.

## The five families

The package groups temporal models into five families, each with a factory function that mirrors the R `causalDeepNet.R` entry points. You can always construct an estimator by its class directly, or through the factory by name.

| Family | Question it answers | Estimators | Factory |
|---|---|---|---|
| **Neural Granger** | Which past series improve the forecast of another? (predictive causality) | `NeuralGrangerCMLP`, `NeuralGrangerCLSTM`, `NeuralGrangerEconomySRU`, `NeuralRelationalInference`, `GrangerLSTM` | `neural_granger_model("cmlp" \| "clstm" \| "economysru" \| "nri")` |
| **Attention / Transformer** | Which variables cause which, and with what delay? | `TCDF`, `CausalTransformer`, `TFTNet` | `attn_causal_model("tcdf" \| "causal_transformer" \| "tft")` |
| **RNN / LSTM** | Interpretable sequential modeling with explicit structure or interventions | `CausalLSTM`, `RETAIN`, `InterventionAwareRNN`, `CausalLSTMForecaster` | `rnn_causal_model("causal_lstm" \| "retain" \| "intervention_rnn" \| "causal_lstm_forecaster")` |
| **Graph neural networks** | A first-class, inspectable causal graph over multivariate series | `GVAR`, `CausalGNN`, `CUTS` | `gnn_causal_model("gvar" \| "causal_gnn" \| "cuts")` |
| **Counterfactual / SCM** | What happens under `do(T)` over a horizon? | `DeepSynth`, `CRN`, `GNet`, `DeepSCM`, `DECI` | `counterfactual_model("deepsynth" \| "crn" \| "gnet")` |

Two more temporal structure-learners sit in `pydeepcausalml.discovery` because they share the NOTEARS machinery of the static discovery methods: **`DynoTEARS`** (lag-resolved DAG discovery) and the acyclicity-constrained variants used by the GNN family.

## Conventions that apply throughout

- **Adjacency orientation.** `A[i, j] = 1` means *X_j causes X_i* — rows are effects/targets, columns are causes/sources.
- **sklearn-style API.** Constructors store hyperparameters; `fit` returns `self`; `history_` records per-epoch diagnostics; discovery models expose `get_adjacency` / `adjacency_matrix` / `get_scores`; forecasters expose `forecast` / `estimate_effect`.
- **Reproducibility.** Pass `random_state=` to any estimator, or call `pydeepcausalml.set_seed`.
- **Validate before you trust.** The `datasets` module (`make_var_data`, `make_intervention_series`, …) provides synthetic series with known ground truth; recover the known graph or effect there before applying a model to real data.

## How to read this chapter

Each family has its own tutorial notebook with runnable code. This introduction gives the shared theory and shows where each model fits; the per-model notebooks then walk through data, fitting, evaluation against ground truth, and interpretation.

---

## Family 1 — Neural Granger Causality

**Estimators:** `NeuralGrangerCMLP`, `NeuralGrangerCLSTM`, `NeuralGrangerEconomySRU`, `NeuralRelationalInference`, `GrangerLSTM` &nbsp;·&nbsp; **Factory:** `neural_granger_model("cmlp" | "clstm" | "economysru" | "nri")`


## Part 1: Conceptual Foundation

### Classical Granger Causality

A variable $X$ Granger-causes $Y$ if past values of $X$ significantly improve the forecast of $Y$ beyond what $Y$'s own past provides. For lag order $p$:

$$
Y_t = \sum_{k=1}^{p} A_k Y_{t-k} + \epsilon_t
$$

If the coefficient block connecting $X_{t-k} \rightarrow Y_t$ is nonzero, then $X$ Granger-causes $Y$. This is inherently linear and can miss nonlinear dynamics.

### Neural Granger Causality: The Big Idea

Replace the linear coefficients with neural networks while preserving sparse input selection logic.

> If a neural network trained to predict $Y_t$ assigns zero (or near-zero) weight to all lags of $X$, then $X$ does not Granger-cause $Y$.

Sparsity is enforced with group LASSO regularization on input weights, grouping all lags of a variable together.

## Part 2: The Four Models

### 1) cMLP (Component-wise MLP)

A separate MLP is trained for each target variable $Y_i$:

$$
\hat{Y}_{i,t} = \text{MLP}_i(Y_{1,t-1:t-p},\ldots,Y_{d,t-1:t-p})
$$

Training objective:

$$
\mathcal{L} = \underbrace{\text{MSE}}_{\text{prediction}} + \lambda \underbrace{\sum_j \lVert W_j^{(1)} \rVert_2}_{\text{group LASSO on input weights}}
$$

### 2) cLSTM (Component-wise LSTM)

Uses one LSTM per target variable. Better for sequential/long-lag dynamics:

$$
h_t = \text{LSTM}_i(Y_{1:d,\ t-p:t-1}), \quad \hat{Y}_{i,t} = W_{\text{out}} h_t
$$

### 3) ECONOMY-SRU

A structurally constrained recurrent unit with a soft gate matrix $G \in [0,1]^{d\times d}$:

- Learns sparse variable interactions
- Lower computational cost than full LSTMs

### 4) NRI (Neural Relational Inference)

NRI (Kipf et al., 2018) is a graph-VAE approach:

- **Encoder** infers latent edge distributions $q(z\mid X)$
- **Decoder** predicts trajectories from sampled edges

$$
\mathcal{L} = \mathbb{E}_{q(z\mid X)}[\log p(X\mid z)] - \mathrm{KL}[q(z\mid X)\|p(z)]
$$

### In PyDeepCausalML

The four theoretical models above map directly onto package estimators, all with a shared `fit(X)` → `get_adjacency()` / `get_scores()` interface:

| Theory | Estimator | Notes |
|---|---|---|
| cMLP | `NeuralGrangerCMLP` | fastest; component-wise MLPs + group LASSO |
| cLSTM | `NeuralGrangerCLSTM` | better for long-lag dynamics |
| Economy-SRU | `NeuralGrangerEconomySRU` | cheap structurally-constrained recurrent unit |
| NRI | `NeuralRelationalInference` | graph-VAE over latent edges |
| full-vs-reduced test | `GrangerLSTM` | ablation-style Granger test |

```python
from pydeepcausalml import NeuralGrangerCMLP, neural_granger_model
from pydeepcausalml.datasets import make_var_data
from pydeepcausalml.metrics import graph_recovery_metrics

X, A_true = make_var_data(n_steps=2000, random_state=0)
cmlp = NeuralGrangerCMLP(lag=5, lambda_group=0.01, random_state=0).fit(X)
print(graph_recovery_metrics(A_true, cmlp.get_adjacency()))
# equivalently: neural_granger_model("cmlp", lag=5).fit(X)
```


---

## Deep Structural Causal Models and lag-resolved discovery

Before the attention, RNN, and GNN families, it helps to see the **structural** view that underlies all of them: a set of equations $X_i = f_i(\mathrm{Pa}(X_i), U_i)$ that can be queried at all three levels of Pearl's hierarchy. Two package components implement this view directly — `DeepSCM` and `DECI` (in `pydeepcausalml.timeseries`) — while `DynoTEARS` (in `pydeepcausalml.discovery`) brings the NOTEARS acyclicity constraint to **time-lagged** discovery.


### Structural Causal Models with deep components

## Part 1: Theory

An SCM is $\mathcal{M}=(\mathbf{S}, P(\mathbf{U}))$ with structural equations

$$X_i = f_i(\mathrm{Pa}(X_i), U_i).$$

SCMs support Pearl's hierarchy:
- Association: $P(Y\mid X=x)$
- Intervention: $P(Y\mid do(X=x))$
- Counterfactual: $P(Y_x\mid X=x', Y=y')$

Deep SCMs replace hand-crafted $f_i$ with neural nets:

$$X_i = f_{\theta_i}(\mathrm{Pa}(X_i), U_i).$$

Variational Deep SCM setup:
- Encoder: $q_\phi(U\mid X)$
- Decoder: $p_\theta(X\mid U,G)$
- ELBO-style objective with reconstruction + KL

DECI jointly learns graph and structural equations with NOTEARS penalty:

$$h(\mathbf{A}) = \mathrm{tr}(e^{\mathbf{A}\odot\mathbf{A}})-d.$$

### In PyDeepCausalML

`DeepSCM` fits neural structural equations on a **fixed** graph and supports interventions via `intervene(...)`; `DECI` learns the graph and the equations **jointly** under a NOTEARS acyclicity penalty and exposes the result through `adjacency_matrix()`.

```python
from pydeepcausalml import DeepSCM, DECI

deci = DECI(random_state=0).fit(X)      # joint graph + structural equations
A = deci.adjacency_matrix()
```


### DYNOTEARS — DAG-Constrained Structure Learning

DYNOTEARS extends NOTEARS-style continuous optimization to **time-series causal discovery** by modeling lagged dependencies while enforcing acyclicity on the contemporaneous graph.

For lag order $p$, a common formulation is:

$$
X_t = W^{(0)}X_t + \sum_{k=1}^{p} W^{(k)}X_{t-k} + \epsilon_t,
$$

where:
- $W^{(0)}$ captures same-time (instantaneous) effects,
- $W^{(k)}$ captures directed lag-$k$ effects,
- acyclicity is imposed on $W^{(0)}$ using a smooth constraint such as NOTEARS:

$$
h(W^{(0)}) = \operatorname{tr}(e^{W^{(0)}\circ W^{(0)}}) - d = 0.
$$

A typical optimization objective combines data fit, sparsity, and DAG constraint:

$$
\min_{\{W^{(k)}\}}\; \mathcal{L}_{\text{recon}} + \lambda_1\sum_{k=0}^{p}\|W^{(k)}\|_1 + \lambda_2 h(W^{(0)}).
$$

Why it is useful in practice:
- Produces **interpretable directed graphs** with explicit lag structure.
- Gives **DAG-guaranteed instantaneous structure** (via acyclicity constraint).
- Works well when you need structure-learning rigor beyond purely predictive models.

Limitations:
- Sensitive to regularization and optimization hyperparameters.
- Optimization can be expensive for high-dimensional or long-lag settings.

**In PyDeepCausalML:** `DynoTEARS` lives in `pydeepcausalml.discovery` (or `causal_structure_ml("dynotears", lag=...)`). Fit it on a multivariate series and read the contemporaneous and lagged structure via `get_adjacency()` / `get_scores()`.

```python
from pydeepcausalml import causal_structure_ml
from pydeepcausalml.datasets import make_var_data

X, A_true = make_var_data(n_steps=2000, random_state=0)
dyno = causal_structure_ml("dynotears", lag=5, random_state=0).fit(X)
A = dyno.get_adjacency()
```


---

## Family 2 — Attention / Transformer Models

**Estimators:** `TCDF`, `CausalTransformer`, `TFTNet` &nbsp;·&nbsp; **Factory:** `attn_causal_model("tcdf" | "causal_transformer" | "tft")`


## Part 1: Theoretical Foundation

`Why Attention for Causality?`

Attention mechanisms compute pairwise relevance scores between all time steps and variables simultaneously. This creates a natural inductive structure for causal discovery:

• Temporal attention → which past time steps influence the present (lag selection)
• Variable attention → which variables influence which (causal graph)
• Causal masking → structural constraint preventing future information leakage

The self-attention score between query $q_i$ and key $k_j$:

$$
 \alpha_{ij} = \frac{\exp\left(q_i^\top k_j / \sqrt{d_k}\right)}{\sum_l \exp\left(q_i^\top k_l / \sqrt{d_k}\right)}
 $$

When interpreted causally, $\alpha_{ij}$ approximates the influence of variable/time $j$ on variable/time $i$.


## Part 2: The Three Models



### TCDF — Temporal Causal Discovery Framework

TCDF (Nauta et al., 2019) uses depthwise separable causal convolutions + self-attention to discover time-delayed causal relationships.

Key ideas:

• `Dilated causal convolutions`: receptive field grows exponentially with depth without data leakage — each output depends only on past inputs
• `Attention mechanism`: scores which input channels (variables) contribute most to each target variable
• `Causal discovery`: for target $Y_i$, trains a separate network; variables with high attention weights are declared causal parents

The convolution at dilation $r$:

$$\text{Conv}(x, t) = \sum_{k=0}^{K-1} w_k \cdot x(t - r \cdot k)$$

Causal mask ensures $x(t - r \cdot k)$ uses only past values ($k \geq 0$, never $k < 0$).

### Causal Transformer

A Transformer with three structural modifications that enforce causal reasoning:

1. `Autoregressive masking`: upper-triangular mask on attention so position $t$ only attends to positions $\leq t$
2. `Inter-variable attention head:` separate attention heads dedicated to cross-variable influence, whose weights are extracted as the causal graph
3. `Positional encoding`: learnable temporal embeddings that encode lag distance

The causal attention output:

$$\text{CausalAttn}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M_{\text{causal}}\right) V$$

where $M_{\text{causal}}[i,j] = -\infty$ if $j > i$, else $0$.

### TFT — Temporal Fusion Transformer

TFT (Lim et al., 2021, Google) is the state-of-the-art architecture for interpretable multi-horizon forecasting. Its causal structure comes from:

• `Variable Selection Networks (VSN)`: soft gating that selects which input variables are relevant at each time step — directly interpretable as variable-level causal weights
• `Gated Residual Networks (GRN)`: nonlinear transformation with skip connections and gating for stable training
• `Multi-head attention with interpretable weights`: attention across time steps, averaged across heads to produce temporal importance scores
• `Quantile output`: produces full predictive distributions, not just point forecasts


TFT architecture flow:

```

Input variables
      │
      ▼
Variable Selection Network  ← learns which vars matter (causal weights)
      │
      ▼
LSTM Encoder/Decoder        ← captures local temporal dynamics
      │
      ▼
Multi-Head Self-Attention   ← captures long-range dependencies
      │
      ▼
Gated Residual + LayerNorm  ← stabilize gradients
      │
      ▼
Quantile Output Head        ← p10, p50, p90 forecasts
```


### In PyDeepCausalML

| Theory | Estimator | Distinctive output |
|---|---|---|
| TCDF | `TCDF` | causal **delays**: `discovered_edges`, `summary()`, `get_adjacency()` |
| Causal Transformer | `CausalTransformer` | attention-derived graph: `get_adjacency()` |
| TFT | `TFTNet` | interpretable multi-horizon forecasts: `predict()` |

TCDF is the go-to when you need **time-delayed** discovery — it reads causal lags off the dilated-convolution kernels after validating candidate causes by permutation importance.

```python
from pydeepcausalml import TCDF
model = TCDF(kernel_size=4, epochs=1000, random_state=0).fit(series)
print(model.summary())        # cause -> effect with estimated delay
A = model.get_adjacency()     # A[i, j] means X_j -> X_i
# equivalently: attn_causal_model("tcdf", kernel_size=4).fit(series)
```


---

## Family 3 — RNN / LSTM Models

**Estimators:** `CausalLSTM`, `RETAIN`, `InterventionAwareRNN`, `CausalLSTMForecaster` &nbsp;·&nbsp; **Factory:** `rnn_causal_model("causal_lstm" | "retain" | "intervention_rnn" | "causal_lstm_forecaster")`


### Conceptual foundation

The RNN/LSTM family covers three approaches for causal time-series modeling:

1. **Causal LSTM** (structure-constrained recurrent modeling)
2. **RETAIN** (reverse-time attention with interpretability)
3. **Intervention-Aware RNN** (regime + intervention conditioning)

The dedicated tutorial notebook works through each on a multivariate financial series; the theory below motivates them.

## Part 1: Theoretical Foundation

### Why RNNs for Causal Time Series?

RNNs process sequences step-by-step, maintaining a hidden state $h_t$ that acts as a compressed memory of the past:

$$h_t = f(W_h h_{t-1} + W_x x_t + b)$$

The hidden state $h_t$ implicitly encodes temporal causal history: which past events shaped the current state. Three architectural extensions turn this into explicit causal machinery.

## Part 2: The Three Models

### 1) Causal LSTM

A standard LSTM augmented with an explicit causal adjacency mask $G \in \{0,1\}^{d \times d}$ that gates which variables are allowed to influence which.

For target variable $i$:

$$\tilde{x}_t^{(i)} = G_{i,:} \odot x_t$$

and the LSTM transitions are computed on $[h_{t-1}; \tilde{x}_t^{(i)}]$. The mask can be fixed, learned (continuous relaxation + sparsity), or hardened at test time.

### 2) RETAIN (Reverse Time Attention Model)

RETAIN uses two attention channels in reverse time order:

- **Temporal attention** $\alpha_t$ (which time step matters)
- **Variable attention** $\beta_t$ (which variable matters at that step)

$$c = \sum_{t=1}^{T} \alpha_t \cdot (\beta_t \odot v_t)$$

A practical attribution score is:

$$\text{Contribution}(j,t) = \alpha_t \cdot |\beta_{t,j}| \cdot |w_j|$$

### 3) Intervention-Aware RNN

To model changing data-generating regimes (policy changes, shocks, treatments), we augment inputs with regime/intervention information:

$$h_t = \text{LSTM}(h_{t-1}, [x_t; r_t; I_t])$$

- $r_t$: learned regime embedding
- $I_t$: intervention indicator

## Interpretation Notes

- **CausalLSTM** yields an explicit, sparse, directed adjacency matrix through a learnable structural mask.
- **RETAIN** gives time- and variable-level attribution, making it useful for interpretability-first workflows.
- **Intervention-Aware RNN** can adapt dynamics when volatility/shock regimes change.

In practice, combining these models is often powerful:
1. Use RETAIN for explanation.
2. Use CausalLSTM for structural constraints.
3. Use Intervention-Aware RNN when regime shifts are expected.

### In PyDeepCausalML

| Theory | Estimator | Use it for |
|---|---|---|
| Causal LSTM | `CausalLSTM` | sequential modeling with a structural mask |
| RETAIN | `RETAIN` | time- and variable-level attribution |
| Intervention-Aware RNN | `InterventionAwareRNN` | regime / intervention conditioning |
| — | `CausalLSTMForecaster` | **counterfactual multi-step forecasting** (`forecast`, `forecast_counterfactual`, `estimate_effect`) |

`CausalLSTMForecaster` is the bridge to the next family: it answers *"what would this series look like under `do(T)`?"* directly.

```python
import numpy as np
from pydeepcausalml import CausalLSTMForecaster

model = CausalLSTMForecaster(pred_len=12, random_state=0)
model.fit(outcome_hist, treat_hist, future)
effect = model.estimate_effect(
    outcome_hist,
    factual_treatment=treat_hist,
    counterfactual_treatment=np.ones_like(treat_hist),   # do(T = 1)
)
print("Mean effect over horizon:", effect.mean())
```


---

## Family 4 — Graph Neural Network Models

**Estimators:** `GVAR`, `CausalGNN`, `CUTS` &nbsp;·&nbsp; **Factory:** `gnn_causal_model("gvar" | "causal_gnn" | "cuts")`


### Conceptual foundation

The GNN family covers three practical approaches to causal modeling for multivariate time series:

1. **GVAR** (Graph Vector Autoregression)
2. **CausalGNN / CD-GNN** (joint graph learning + dynamics)
3. **CUTS+ inspired model** (causal discovery under missing data)

The dedicated tutorial notebook demonstrates them on sector-ETF returns in pure PyTorch (no mandatory `torch-geometric`); the theory below sets them up.

## Part 1: Theoretical Foundation

### Why GNNs for Causal Discovery?

A GNN treats variables as nodes and causal relationships as edges in a graph

$$\mathcal{G}=(\mathcal{V},\mathcal{E}).$$

Message passing makes the causal structure explicit and inspectable:

$$
\mathbf{h}_i^{(\ell+1)}=\mathrm{UPDATE}\!\left(\mathbf{h}_i^{(\ell)},\ \mathrm{AGGREGATE}\left(\{\mathbf{h}_j^{(\ell)}:j\in\mathcal{N}(i)\}\right)\right).
$$

Unlike RNNs/Transformers where causal effects are implicit in hidden states, in GNN causal models the graph itself is a first-class object that can be regularized, inspected, and interpreted.

## Part 2: The Three Models

### 1) GVAR — Graph Vector Autoregression

GVAR extends classical VAR with a learned graph and nonlinear feature transforms:

$$
X_i^{(t)}=\sum_{k=1}^{p}\sum_{j=1}^{d}A_{ij}^{(k)}\,\phi\!\left(X_j^{(t-k)}\right)+\epsilon_i^{(t)}.
$$

- $A^{(k)}\in\mathbb{R}^{d\times d}$: lag-specific adjacency (learned)
- $\phi(\cdot)$: nonlinear transform (GNN feature map)
- sparsity via L1/group-LASSO style penalties

### 2) CausalGNN / CD-GNN

Two-stage joint learning:

1. **Graph learner** outputs soft adjacency $\hat{G}$ with NOTEARS acyclicity

$$
h(\mathbf{A})=\mathrm{tr}(e^{\mathbf{A}\odot\mathbf{A}})-d=0
$$

2. **Dynamics learner** propagates messages along parents:

$$
\mathbf{m}_{j\rightarrow i}^{(t)}=\mathrm{MSG}(\mathbf{h}_j^{(t)},\mathbf{h}_i^{(t)},e_{ij}),\quad
\mathbf{h}_i^{(t+1)}=\mathrm{UPDATE}\!\left(\mathbf{h}_i^{(t)},\sum_{j\in\mathrm{Pa}(i)}\mathbf{m}_{j\rightarrow i}^{(t)}\right).
$$

### 3) CUTS / CUTS+

Designed for unevenly sampled or missing time series:

- causal-aware imputation network
- variational posterior over graphs: $q(G)\approx\prod_{ij}\mathrm{Bernoulli}(\pi_{ij})$
- joint optimization of reconstruction + imputation + graph regularization

A common ELBO form is:

$$
\mathcal{L}=\mathbb{E}[\log p(X_{\mathrm{obs}}\mid G,X_{\mathrm{miss}})]
+\mathbb{E}[\log p(X_{\mathrm{miss}}\mid G)]
-\mathrm{KL}[q(G)\|p(G)].
$$

## Part 3: Model 1 — GVAR (Graph VAR)

This model learns lag-specific adjacency matrices jointly with nonlinear temporal dynamics.

**In PyDeepCausalML:** `GVAR` (or `gnn_causal_model("gvar")`) learns the lag-specific adjacency and exposes it through `causal_matrix()`.


## Part 4: Model 2 — CausalGNN / CD-GNN

This model learns a soft DAG and uses edge-conditioned message passing for forecasting.

**In PyDeepCausalML:** `CausalGNN` (or `gnn_causal_model("causal_gnn")`) returns the learned soft DAG through `causal_matrix()`.


## Part 5: Model 3 — CUTS+ Inspired (Missing Data)

A practical approximation of CUTS+ ideas: learn probabilistic graph edges and jointly impute missing values before forecasting.

**In PyDeepCausalML:** `CUTS` (or `gnn_causal_model("cuts")`) targets irregular / missing series and exposes the recovered graph through `causal_matrix()`.


### In PyDeepCausalML

All three GNN models share a `fit(X)` → `causal_matrix()` interface and can be built by class or factory:

```python
from pydeepcausalml import GVAR, gnn_causal_model

gvar = GVAR(lag=5, random_state=0).fit(X)
A = gvar.causal_matrix()
# equivalently: gnn_causal_model("gvar", lag=5).fit(X)
```

Use the GNN family when you want the causal graph itself to be a **first-class, regularizable object** — `GVAR` for lag-specific linear-plus-nonlinear structure, `CausalGNN` for a NOTEARS-constrained soft DAG with message passing, and `CUTS` for irregular or missing series.


---

## Family 5 — Counterfactual / SCM Forecasting

**Estimators:** `DeepSynth`, `CRN`, `GNet`, `DeepSCM`, `DECI` &nbsp;·&nbsp; **Factory:** `counterfactual_model("deepsynth" | "crn" | "gnet")`

The first four families answer *structural* questions — which series drives which, at what lag. This family answers the *interventional* question that motivates much applied time-series causal work: **given the history, what would the outcome trajectory have been under a different treatment path?**


### Why a separate family?

Counterfactual forecasting has to handle **time-varying confounding**: a treatment decision at time $t$ depends on the history up to $t$, and also affects future covariates that in turn drive later treatment decisions. Naively regressing outcome on treatment history is biased. These models borrow ideas from the g-methods and representation balancing to break that feedback loop.

| Model | Idea | Method |
|---|---|---|
| `DeepSynth` | deep synthetic control — reconstruct the untreated trajectory from donors | `predict_counterfactual` |
| `CRN` | counterfactual recurrent network with balanced representations across treatments | `predict_ite` |
| `GNet` | neural g-computation: simulate trajectories forward under a treatment rule | `predict_ite` |
| `DeepSCM` | neural structural equations, intervene on the SCM directly | `intervene` |
| `DECI` | learn graph **and** equations jointly, then intervene | `adjacency_matrix`, intervene |

The **key output** across this family is a *counterfactual trajectory* or an *effect over a horizon*, not a single scalar ATE.

```python
from pydeepcausalml import counterfactual_model
from pydeepcausalml.datasets import make_intervention_series

df = make_intervention_series(n_units=500, horizon=24, random_state=0)
crn = counterfactual_model("crn", random_state=0).fit(df)
ite = crn.predict_ite(df)      # per-unit, per-horizon effect
```


---

## Chapter map

| If your question is… | Reach for | Family |
|---|---|---|
| "Which past series improves the forecast of another?" | `NeuralGrangerCMLP` (fast), `NeuralGrangerCLSTM` | Neural Granger |
| "Which series drives which, **and at what lag**?" | `TCDF` (delays), `DynoTEARS` (DAG guarantee) | Attention / Discovery |
| "Interpretable forecast with variable/time attribution" | `RETAIN`, `TFTNet` | RNN / Attention |
| "A regularizable causal graph over many series" | `GVAR`, `CausalGNN`, `CUTS` | GNN |
| "What would this series look like under `do(T)`?" | `CausalLSTMForecaster`, `CRN`, `GNet`, `DeepSynth` | Counterfactual / SCM |

Each family has a dedicated tutorial notebook that fits the models on realistic data, checks them against ground truth from the `datasets` module, and interprets the recovered structure or counterfactuals. Start with the family that matches your question, and use the factory functions (`neural_granger_model`, `attn_causal_model`, `rnn_causal_model`, `gnn_causal_model`, `counterfactual_model`) to swap models within a family without changing the surrounding code.

> **A note on validation.** Every estimator here can — and should — be checked on synthetic data with known ground truth before you trust it on real series. Discovery models should score well on `graph_recovery_metrics`; effect and forecasting models should recover the known effect within noise. If they don't at your data's scale and noise level, tune before drawing conclusions.
